# Monte Carlo — refined sleeves **and** the research-stack books

Two systems live in this repo and they do not share a unit. This notebook now
covers both, in that order, and never mixes them without saying so.

## Part A (§1–6) — the **legacy engine**, ablation-refined sleeves

Bootstrap risk analysis of the configurations that survived the 2026-07-18
full-matrix ablations (`ablate.py` + `ablate_vbt.py`):

| Sleeve | Refined rules | Why |
|---|---|---|
| residualmomentum | RA-RES only | all 3 confirmation legs are drag |
| macd | CROSS + STAGE=sub0 | EMA50 / RSI dead weight |
| bollinger | full 4-leg, **choppy regime only** | its routing slot; all-regime view misleads |
| vwap | AVWAP + VOL + RSI-RESET | EMA-STACK mildly dead |
| pead | event gate only | no-rules baseline beats every combo |
| consolidation | — retired | COMP leg toxic |

Each sleeve's trades come from the vectorized VBT backtest
(`rhengine/mc_refined.py`, same fills/exits/costs as `ablate_vbt.py`, cost
resolved from `config.json` `costs.round_trip_bps` **round trip**, 730d
full-market Massive cache). Per-trade **R-multiples**
(trade return ÷ stop-distance fraction at entry) are resampled with
replacement into thousands of equity paths under fixed-fractional sizing —
so one lucky trade *ordering* can't hide tail risk.

## Part B (§7–9) — the **research stack**, RMX-B and RMX-C

| Book | What it is |
|---|---|
| **RMX-B** | the shipped strategy: 50/50 cap-weighted quality (OP/BE, June-formed) + 12-1 momentum (monthly), 10% name cap, ~96 names. Vol-scaling **off** |
| **RMX-C** | the live extension: 5 blocks — quality, momentum, investment, issuance, payout — as one netted vector, ~154 names. Momentum block vol-scaled **on** |

These have no trades and no R-multiples. They are daily mark-to-market books
(`rhengine/alpha/book.py`), so they get their own bootstrap
(`rhengine/mc_books.py`): stationary **blocks** over sessions, fully invested,
with SPY resampled on the *same* block indices. §9 puts all of it in one table
with an explicit `unit` column, because the `final ×` figures from Part A and
Part B are not comparable and the table would otherwise invite exactly that
mistake.

**Run top-to-bottom** (VS Code: open and *Run All*; after editing `rhengine/`,
Restart Kernel first). Missing caches are harvested automatically — Part A
takes several minutes on the full market, Part B about ten seconds.

In [ ]:
from pathlib import Path
import subprocess, sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "config.json").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from rhengine.mc_refined import run_mc_paths, REFINED_SET
from rhengine.mc_books import BOOK_SERIES, run_mc_book_paths
from rhengine.alpha.stats import perf
from rhengine.costs import default_cost_bps

# ---- parameters, legacy engine (trade-space) ---------------------------
# Audit FC-2: config.json costs.round_trip_bps (30.0), not a literal 5.
COST_BPS   = default_cost_bps(ROOT / "config.json")
# Cache is cost-stamped: a 5 bps harvest must never be reused at 30 bps.
TRADES_CSV = ROOT / f"tests/results/refined_trades_{COST_BPS:g}bps.csv"
REFRESH    = False      # True -> re-run the harvest even if the CSV exists
PATHS      = 20_000     # bootstrap paths (findings spec: >= 20k)
N_TRADES   = 250        # trades per path (~a year of signals per sleeve)
RISK_PCT   = 1.0        # equity % risked per trade (1R)
RUIN_DD    = 50.0       # drawdown % treated as "ruin"
SEED       = 1337

# ---- parameters, research-stack books (returns-space) ------------------
# DIFFERENT UNIT — see section 7. RMX-B and RMX-C are daily mark-to-market
# books with no trades, no R-multiples and no risk-per-trade dial, so they are
# block-bootstrapped over SESSIONS.
#
# WINDOW WIDENED 2026-08-01, and the reason matters more than the dates.
# This was 2010-07..2018-12, the in-sample window — so the bootstrap resampled
# only the stretch in which the books worked and EXCLUDED 2019-2026, in which
# they trailed. It reported RMX-C beating SPY in 61% of resampled years. On the
# full sample the identical computation reports 47%. A win rate drawn from a
# window that omits the losing years is not a statistic, it is a selection.
#
# The full window is the in-sample period plus the SPENT holdout glued together
# (CLAUDE.md rule 4). It is not out-of-sample anything, the look is logged in
# docs/HOLDOUT.md, and no parameter may move in response to it. Set
# BOOK_HI = "2018-12-31" to reproduce the older, narrower figures.
BOOK_LO, BOOK_HI = "2010-07-01", "2026-07-31"

# The cost model. 'flat' charges ONE number to every name — what the published
# figures assumed. 'tier-*' charges each name its own market-cap tier's spread
# (costs.CAP_TIERS), which is the honest model: these books are >92% mega-cap
# and pay ~2.8 bps round trip, so the legacy scan universe's flat 30 overstates
# their cost ~10x. Research rules §6a. Run 'tier-lo' and 'tier-hi' as a bracket.
# The cache is stamped with the MODEL, so the two never mix.
BOOK_COST_MODEL = "tier-point"    # flat | tier-lo | tier-point | tier-hi
BOOK_COST_BPS   = 10.0            # only used when BOOK_COST_MODEL == "flat"
_stamp          = (f"{BOOK_COST_BPS:g}bps" if BOOK_COST_MODEL == "flat"
                   else BOOK_COST_MODEL)
BOOK_CSV        = ROOT / f"tests/results/book_returns_{_stamp}.csv"
BOOK_REFRESH    = False
HORIZON         = 252   # sessions per path for the fan charts (one year)
MEAN_BLOCK      = 21    # ~one month per block; the validation.py convention
# One year is the wrong unit for a book whose whole claim is a small edge
# compounding: at 252 sessions the market's own noise is 20-50x the signal, so
# a near-coin-flip is the only answer arithmetic permits. §8.2 sweeps the
# holding period instead, which is where the question actually gets answered.
SWEEP_HORIZONS  = (252, 756, 1260, 2520)      # 1 / 3 / 5 / 10 years

# ---- palette (dataviz reference, light mode; fixed slot order) ---------
HUES = {"residualmomentum": "#2a78d6", "macd": "#008300",
        "bollinger": "#e87ba4", "vwap": "#eda100", "pead": "#1baf7a",
        "POOLED": "#0b0b0b",
        # research-stack books + their benchmarks, deliberately a separate
        # part of the wheel so a book is never mistaken for a legacy sleeve
        "RMX-B": "#6b4bd6", "RMX-C": "#c25e00",
        "SPY": "#898781", "CW universe": "#c3c2b7"}
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "figure.dpi": 110,
    "font.family": "Segoe UI", "font.size": 10,
    "text.color": INK, "axes.labelcolor": INK2,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": BASELINE, "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlecolor": INK, "axes.titlesize": 11,
    "legend.frameon": False,
})

## 1 · Load the refined-set trades

In [ ]:
if REFRESH or not TRADES_CSV.exists():
    print("Harvesting refined-set trades (full market, ~minutes)...")
    subprocess.run([sys.executable, "-m", "rhengine.mc_refined",
                    "--lookback-days", "730", "--cost-bps", str(COST_BPS),
                    "--csv", str(TRADES_CSV)], cwd=ROOT, check=True)

trades = pd.read_csv(TRADES_CSV, parse_dates=["entry_date", "exit_date"])
ORDER = [s for s in REFINED_SET if s in set(trades["strategy"])]

summary = (trades.groupby("strategy")
           .agg(rules=("rules", "first"), n=("r_multiple", "size"),
                win_rate=("r_multiple", lambda r: (r > 0).mean()),
                avg_R=("r_multiple", "mean"), med_R=("r_multiple", "median"),
                p95_R=("r_multiple", lambda r: r.quantile(0.95)))
           .reindex(ORDER).round(3))
print(f"{len(trades):,} trades, "
      f"{trades.entry_date.min().date()} - {trades.entry_date.max().date()}")
summary

## 2 · Bootstrap the equity paths

For each sleeve (and the pooled set) we resample `N_TRADES` R-multiples with
replacement, `PATHS` times, compounding at `RISK_PCT`% risk per trade. Kept
per run: the percentile **fan** of the equity path, plus the final-equity and
max-drawdown distributions.

In [ ]:
FAN_PCTS = (5, 25, 50, 75, 95)
mc, fans = {}, {}
pools = {s: trades.loc[trades.strategy == s, "r_multiple"].values for s in ORDER}
pools["POOLED"] = trades["r_multiple"].values

for name, rs in pools.items():
    r = run_mc_paths(rs, n_trades=N_TRADES, paths=PATHS,
                     risk_pct=RISK_PCT, seed=SEED, ruin_dd_pct=RUIN_DD)
    fans[name] = np.percentile(r.pop("equity"), FAN_PCTS, axis=0)
    mc[name] = r

risk_tbl = pd.DataFrame({
    name: {"observed trades": r["n_observed"], "avg R": round(r["avg_r"], 3),
           "final ×  p5": round(r["final_pct"][5], 2),
           "final ×  p50": round(r["final_pct"][50], 2),
           "final ×  p95": round(r["final_pct"][95], 2),
           "maxDD %  p50": round(r["maxdd_pct"][50], 1),
           "maxDD %  p95": round(r["maxdd_pct"][95], 1),
           "P(loss) %": round(r["prob_loss"], 1),
           f"P(DD>{RUIN_DD:.0f}%) %": round(r["prob_ruin"], 1)}
    for name, r in mc.items()}).T
risk_tbl

## 3 · Equity fan charts

Each panel: one sleeve's bootstrap paths as percentile bands — outer band
p5–p95, inner p25–p75, heavy line = median — over 250 resampled trades at
the configured risk. The dashed hairline is break-even (1.0×).

In [ ]:
names = ORDER + ["POOLED"]
fig, axes = plt.subplots(2, 3, figsize=(12.5, 6.8), sharex=True)
x = np.arange(1, N_TRADES + 1)
for ax, name in zip(axes.flat, names):
    hue, fan = HUES[name], fans[name]
    ax.fill_between(x, fan[0], fan[4], color=hue, alpha=0.14, lw=0)
    ax.fill_between(x, fan[1], fan[3], color=hue, alpha=0.28, lw=0)
    ax.plot(x, fan[2], color=hue, lw=2)
    ax.axhline(1.0, color=BASELINE, lw=1, ls=(0, (4, 3)))
    for row, p in ((0, "p5"), (2, "p50"), (4, "p95")):
        ax.annotate(f" {p} {fan[row, -1]:.2f}×", (x[-1], fan[row, -1]),
                    fontsize=8, color=INK2 if row != 2 else INK, va="center")
    ax.set_title(name if name == "POOLED" else
                 f"{name} · {trades.loc[trades.strategy == name, 'rules'].iat[0]}",
                 loc="left", fontsize=9.5)
    ax.set_xlim(0, N_TRADES * 1.16)
    ax.margins(y=0.06)
for ax in axes[1]:
    ax.set_xlabel("trade #")
for ax in axes[:, 0]:
    ax.set_ylabel("equity multiple")
fig.suptitle(f"Bootstrap equity fans — {PATHS:,} paths × {N_TRADES} trades, "
             f"{RISK_PCT:.1f}% risk/trade", x=0.01, ha="left",
             fontsize=12, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

## 4 · Median paths, head-to-head

One axis, the median bootstrap path of every sleeve. This is the *typical*
outcome — the fans above show how wide "typical" really is.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.6))
for name in names:
    med = fans[name][2]
    ax.plot(x, med, color=HUES[name], lw=2,
            ls=(0, (5, 2)) if name == "POOLED" else "-")
    ax.annotate(f" {name} {med[-1]:.2f}×", (x[-1], med[-1]),
                color=HUES[name] if name != "vwap" else "#c98500",
                fontsize=9, va="center", fontweight="bold")
ax.axhline(1.0, color=BASELINE, lw=1, ls=(0, (4, 3)))
ax.set_xlim(0, N_TRADES * 1.42)
ax.set_xlabel("trade #"); ax.set_ylabel("median equity multiple")
ax.set_title(f"Median bootstrap path per sleeve ({RISK_PCT:.1f}% risk/trade)",
             loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

## 5 · Outcome distributions

Left: where the paths **end** (final equity multiple). Right: the **worst
drawdown** each path suffered on the way. Vertical hairline: break-even /
the ruin threshold.

In [ ]:
fig, axes = plt.subplots(len(names), 2, figsize=(11, 1.75 * len(names)),
                         sharex="col")
for i, name in enumerate(names):
    hue, r = HUES[name], mc[name]
    axL, axR = axes[i]
    axL.hist(np.clip(r["finals"], 0, np.percentile(r["finals"], 99.5)),
             bins=60, color=hue, alpha=0.85)
    axL.axvline(1.0, color=INK2, lw=1, ls=(0, (4, 3)))
    axL.annotate(f"{name}  ·  P(loss) {r['prob_loss']:.0f}%",
                 (0.99, 0.82), xycoords="axes fraction", ha="right",
                 fontsize=9, color=INK, fontweight="bold")
    axR.hist(r["maxdd"] * 100, bins=60, color=hue, alpha=0.85)
    axR.axvline(RUIN_DD, color="#d03b3b", lw=1, ls=(0, (4, 3)))
    axR.annotate(f"p95 {r['maxdd_pct'][95]:.0f}%  ·  "
                 f"P(DD>{RUIN_DD:.0f}%) {r['prob_ruin']:.1f}%",
                 (0.99, 0.82), xycoords="axes fraction", ha="right",
                 fontsize=9, color=INK2)
    for a in (axL, axR):
        a.set_yticks([])
axes[-1, 0].set_xlabel("final equity multiple")
axes[-1, 1].set_xlabel("max drawdown %")
fig.suptitle("Final equity (left) and max drawdown (right) across paths",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()

## 6 · How hot can the risk dial go?

The same pooled bootstrap swept across risk-per-trade settings. Two panels,
one measure each: growth (median / p5 final multiple) and pain (p95 max
drawdown, with the ruin threshold marked). Pick the largest risk whose
**pain** you can actually sit through — the growth panel alone always argues
for more.

In [ ]:
risks = np.arange(0.25, 3.01, 0.25)
sweep = [run_mc_paths(pools["POOLED"], n_trades=N_TRADES, paths=5_000,
                      risk_pct=rp, seed=SEED, ruin_dd_pct=RUIN_DD)
         for rp in risks]
med_f = [s["final_pct"][50] for s in sweep]
p5_f  = [s["final_pct"][5] for s in sweep]
p95dd = [s["maxdd_pct"][95] for s in sweep]

blue = HUES["residualmomentum"]
fig, (axG, axP) = plt.subplots(1, 2, figsize=(11, 4.2))
axG.plot(risks, med_f, color=blue, lw=2, marker="o", ms=4)
axG.plot(risks, p5_f, color=blue, lw=2, marker="o", ms=4, ls=(0, (5, 2)))
axG.annotate(" median", (risks[-1], med_f[-1]), color=blue,
             fontsize=9, va="center", fontweight="bold")
axG.annotate(" p5 (bad luck)", (risks[-1], p5_f[-1]), color=INK2,
             fontsize=9, va="center")
axG.axhline(1.0, color=BASELINE, lw=1, ls=(0, (4, 3)))
axG.set_xlim(0.1, 3.75)
axG.set_xlabel("risk per trade %"); axG.set_ylabel("final equity multiple")
axG.set_title("Growth", loc="left", fontweight="bold")

axP.plot(risks, p95dd, color="#d03b3b", lw=2, marker="o", ms=4)
axP.axhline(RUIN_DD, color="#d03b3b", lw=1, ls=(0, (4, 3)))
axP.annotate(f"ruin threshold {RUIN_DD:.0f}%", (0.3, RUIN_DD),
             fontsize=8.5, color="#d03b3b", va="bottom")
axP.set_xlabel("risk per trade %"); axP.set_ylabel("p95 max drawdown %")
axP.set_title("Pain", loc="left", fontweight="bold")
fig.suptitle(f"Pooled refined set — risk sweep ({N_TRADES} trades/path)",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

---

# 7 · The research-stack books — RMX-B and RMX-C

Everything above this line is the **legacy engine**: per-trade R-multiples,
resampled i.i.d., sized at a chosen % of equity per trade. RMX-B and RMX-C are
not that. They are the *other* system in this repo — cross-sectional
rank-and-hold books over the point-in-time S&P 500, whose only output is a
**daily mark-to-market return series** (`rhengine/alpha/book.py`, the only
place a performance number in this project may come from).

**Read the units before the charts.**

| | sections 1–6 | this section |
|---|---|---|
| code | `rhengine/*.py` | `rhengine/alpha/*.py` |
| one draw is | one **trade** | one **session** |
| resampling | i.i.d. | stationary **blocks**, mean 21 sessions |
| sizing | `RISK_PCT`% of equity per trade | none — weights sum to 1, always |
| "P(loss)" means | the strategy lost money | a fully invested equity book was down over the horizon — SPY's own figure on the same resampling sits beside it, and the two barely differ |
| the number that matters | P(loss), max DD | **P(underperform SPY)**, swept over horizons in §8.2 |

Blocks rather than i.i.d. days is not a stylistic choice. Daily equity returns
cluster — volatility arrives in runs — and i.i.d. resampling destroys exactly
the structure that produces real drawdowns (audit FC-10). The scheme is Politis
& Romano's stationary bootstrap, the same one `alpha/validation.py` uses; this
module keeps the full path matrix so the fans can be drawn.
`tests/test_mc_books.py::test_blocks_preserve_loss_clustering` asserts the
difference is real.

**The window, and why it changed.** Until 2026-08-01 this section resampled
2010-07 .. 2018-12 only — the in-sample window. That excluded 2019–2026, the
stretch in which both books actually trailed SPY, and it reported RMX-C winning
**61%** of resampled years. On the full 2010–2026 sample the identical
computation reports **47%**. Nothing about the strategy changed; the sample
stopped omitting the losing years. Treat that as the single most important
number in this notebook, and as a standing warning about every bootstrap whose
sample you did not choose deliberately.

**What this still cannot do.** It re-arranges one realised sample, and that
sample is now the in-sample period plus the SPENT holdout glued together — it
is not out-of-sample anything, the look is logged in `docs/HOLDOUT.md`, and no
parameter may move in response to it. RMX-B's actual evidence remains Ken
French's 63 years (+1.78%/yr alpha, t = +2.54); against a 5.5%/yr tracking
error that needs ~39 years to reach t = 2, so 16 years here can check the
implementation for *consistency* and never for *significance*. Read the fans as
the shape of the risk.

In [ ]:
import json

if BOOK_REFRESH or not BOOK_CSV.exists():
    print("Harvesting RMX-B / RMX-C daily book returns (~15s)...")
    subprocess.run([sys.executable, "-m", "rhengine.mc_books",
                    "--lo", BOOK_LO, "--hi", BOOK_HI,
                    "--cost-bps", str(BOOK_COST_BPS),
                    "--cost-model", BOOK_COST_MODEL,
                    "--csv", str(BOOK_CSV)], cwd=ROOT, check=True)

books = pd.read_csv(BOOK_CSV, index_col=0, parse_dates=True)
# The data stamp. `Panel.fingerprint` is window-scoped, so the weekly price
# append leaves it alone and a change means the BACKTEST changed. Quote it with
# any number taken from below — docs/RMX-C.md records what happens otherwise.
meta = json.loads(BOOK_CSV.with_suffix(".meta.json").read_text())
eff = meta.get("effective_bps", {})
print(f"{len(books):,} sessions, {books.index[0].date()} - "
      f"{books.index[-1].date()}  ·  cost model {meta['cost_model']}")
print(f"panel {meta['panel_fp']}  ·  harvested {meta['harvested_at'][:10]}")
print("effective round trip  " + "  ".join(f"{k} {v:.2f} bps"
                                           for k, v in eff.items())
      + "   |   turnover/yr  "
      + "  ".join(f"{k} {v * 100:.0f}%"
                  for k, v in meta["turnover_ann"].items()))

# The REALISED record first. The bootstrap below cannot mean anything until you
# have seen what it is resampling — and the gap between RMX-B/RMX-C and SPY
# here is 0.7-0.9 pp/yr on 8.5 years, i.e. nowhere near significant locally.
book_perf = pd.DataFrame(
    [{"CAGR %": (p := perf(books[c], books["rf"], c)).cagr_pct,
      "vol %": p.vol_ann_pct, "Sharpe": p.sharpe, "maxDD %": p.max_dd_pct,
      "worst month %": p.worst_month_pct, "skew": p.skew,
      "hit months %": p.hit_months_pct}
     for c in BOOK_SERIES],
    index=list(BOOK_SERIES)).round(2)
book_perf

## 8 · Block bootstrap — the books

`HORIZON` sessions per path, `PATHS` paths, geometric blocks averaging
`MEAN_BLOCK` sessions. SPY is resampled with the **same block indices** as each
book, so `P(< SPY)` compares the book to a market path that actually co-moves
with it — resampling them independently would compare it to a market that never
happened.

In [ ]:
bmc, bfans = {}, {}
for name in BOOK_SERIES:
    r = run_mc_book_paths(books[name], horizon=HORIZON, paths=PATHS,
                          mean_block=MEAN_BLOCK, seed=SEED,
                          ruin_dd_pct=RUIN_DD,
                          bench=None if name == "SPY" else books["SPY"])
    bfans[name] = np.percentile(r.pop("equity"), FAN_PCTS, axis=0)
    bmc[name] = r

book_tbl = pd.DataFrame({
    name: {"observed sessions": r["n_observed"],
           "realised CAGR %": round(r["realised_cagr_pct"], 2),
           "final ×  p5": round(r["final_pct"][5], 2),
           "final ×  p50": round(r["final_pct"][50], 2),
           "final ×  p95": round(r["final_pct"][95], 2),
           "maxDD %  p50": round(r["maxdd_pct"][50], 1),
           "maxDD %  p95": round(r["maxdd_pct"][95], 1),
           "P(down yr) %": round(r["prob_loss"], 1),
           "P(< SPY) %": ("—" if np.isnan(r["prob_underperform"])
                          else round(r["prob_underperform"], 1)),
           f"P(DD>{RUIN_DD:.0f}%) %": round(r["prob_ruin"], 1)}
    for name, r in bmc.items()}).T
book_tbl

In [ ]:
bx = np.arange(1, HORIZON + 1)
fig, axes = plt.subplots(1, 4, figsize=(13.5, 3.9), sharey=True)
for ax, name in zip(axes, BOOK_SERIES):
    hue, fan = HUES[name], bfans[name]
    ax.fill_between(bx, fan[0], fan[4], color=hue, alpha=0.14, lw=0)
    ax.fill_between(bx, fan[1], fan[3], color=hue, alpha=0.28, lw=0)
    ax.plot(bx, fan[2], color=hue, lw=2)
    # SPY's median on every panel: a long-only large-cap book lives or dies
    # against it, and a fan drawn without it flatters whatever it contains
    if name != "SPY":
        ax.plot(bx, bfans["SPY"][2], color=INK2, lw=1, ls=(0, (5, 2)))
    ax.axhline(1.0, color=BASELINE, lw=1, ls=(0, (4, 3)))
    for row, p in ((0, "p5"), (2, "p50"), (4, "p95")):
        ax.annotate(f" {p} {fan[row, -1]:.2f}×", (bx[-1], fan[row, -1]),
                    fontsize=8, color=INK if row == 2 else INK2, va="center")
    sub = "" if name == "SPY" else f"  ·  P(<SPY) {bmc[name]['prob_underperform']:.0f}%"
    ax.set_title(f"{name}{sub}", loc="left", fontsize=9.5)
    ax.set_xlim(0, HORIZON * 1.30)
    ax.set_xlabel("trading day")
axes[0].set_ylabel("equity multiple")
fig.suptitle(f"Research-stack books — block bootstrap, {PATHS:,} paths × "
             f"{HORIZON} sessions, mean block {MEAN_BLOCK}d "
             f"(dashed = SPY median)", x=0.01, ha="left",
             fontsize=12, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.92))
plt.show()

### 8.1 · The only book chart that is about the strategy

The fans above are ~95% market. A long-only large-cap book with beta ≈ 1 will
always produce a handsome fan, because the *market* produces a handsome fan —
the shipped RMX-B is +1.78%/yr of alpha, and at this scale that is a hairline.

So subtract the paired market path. Left: the book's excess over SPY on the
**same** resampled year, in percentage points. Right: the max-drawdown
distribution, which is the number that decides whether you can hold it.

Read the left panel honestly: the mass sitting either side of zero is the
"expect to trail the index roughly 4 years in 10" warning, drawn.

In [ ]:
rel_names = [n for n in BOOK_SERIES if n != "SPY"]
fig, axes = plt.subplots(len(rel_names), 2, figsize=(11, 2.3 * len(rel_names)),
                         sharex="col")
for i, name in enumerate(rel_names):
    hue, r = HUES[name], bmc[name]
    axL, axR = axes[i]
    excess = (r["finals"] - r["bench_finals"]) * 100      # pp over SPY, paired
    axL.hist(np.clip(excess, *np.percentile(excess, [0.25, 99.75])),
             bins=70, color=hue, alpha=0.85)
    axL.axvline(0.0, color=INK2, lw=1, ls=(0, (4, 3)))
    axL.set_title(f"{name} — beats SPY in {100 - r['prob_underperform']:.0f}% "
                  f"of resampled years · median {np.median(excess):+.2f} pp",
                  loc="left", fontsize=9.5, fontweight="bold", color=hue)
    axR.hist(r["maxdd"] * 100, bins=70, color=hue, alpha=0.85)
    axR.axvline(RUIN_DD, color="#d03b3b", lw=1, ls=(0, (4, 3)))
    axR.set_title(f"max drawdown — p50 {r['maxdd_pct'][50]:.0f}% · p95 "
                  f"{r['maxdd_pct'][95]:.0f}% · p99 {r['maxdd_pct'][99]:.0f}%",
                  loc="left", fontsize=9.5, color=INK2)
    for a in (axL, axR):
        a.set_yticks([])
axes[-1, 0].set_xlabel("excess over SPY on the same resampled year (pp)")
axes[-1, 1].set_xlabel("max drawdown %")
fig.suptitle("Paired excess over SPY (left) and max drawdown (right)",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

### 8.2 · Sweep the holding period — where the question actually gets answered

One year is the wrong unit. A ~0.5%/yr edge inside a ~17%/yr swing is a
signal-to-noise ratio around 1:30, so **at a one-year horizon a coin flip is
the only answer arithmetic permits** — and that is exactly what §8.1 shows.

The textbook hope is that time fixes this: edge grows with **T**, noise grows
with **√T**, so the win rate should climb with the holding period. Sweeping
1 / 3 / 5 / 10 years tests whether that actually happens here.

Watch the direction of the `beats SPY` column as the horizon lengthens. If it
rises, there is a small persistent edge that patience harvests. If it falls,
the book is losing to the benchmark systematically and time makes it worse,
not better — because the same arithmetic that would compound an edge also
compounds a deficit.

In [ ]:
sweep_rows, beats = [], {n: [] for n in BOOK_SERIES if n != "SPY"}
for h in SWEEP_HORIZONS:
    for name in BOOK_SERIES:
        r = run_mc_book_paths(books[name], horizon=h, paths=PATHS,
                              mean_block=MEAN_BLOCK, seed=SEED,
                              ruin_dd_pct=RUIN_DD,
                              bench=None if name == "SPY" else books["SPY"])
        beat = (np.nan if np.isnan(r["prob_underperform"])
                else 100 - r["prob_underperform"])
        if name in beats:
            beats[name].append(beat)
        sweep_rows.append({"years": h / 252, "book": name,
                           "final ×  p5": round(r["final_pct"][5], 2),
                           "final ×  p50": round(r["final_pct"][50], 2),
                           "final ×  p95": round(r["final_pct"][95], 2),
                           "maxDD %  p50": round(r["maxdd_pct"][50], 1),
                           "maxDD %  p95": round(r["maxdd_pct"][95], 1),
                           "beats SPY %": ("—" if np.isnan(beat)
                                           else round(beat, 1))})
sweep = pd.DataFrame(sweep_rows).set_index(["years", "book"])

# Overlap warning, stated rather than buried: a path longer than half the
# sample reuses most of it, so long-horizon SPREADS are understated. The
# central tendency and the win rate are still readable.
long_h = [h for h in SWEEP_HORIZONS if h > len(books) // 2]
if long_h:
    print(f"NOTE: horizons {long_h} exceed half the {len(books)}-session "
          f"sample; paths overlap heavily\nand their p5/p95 spread is "
          f"UNDERSTATED. Read the medians and the win rate, not the tails.")
sweep

In [ ]:
yrs = [h / 252 for h in SWEEP_HORIZONS]
fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.axhspan(0, 50, color="#d03b3b", alpha=0.05, lw=0)
ax.axhline(50, color=INK2, lw=1, ls=(0, (4, 3)))
ax.annotate(" coin flip", (yrs[-1] * 1.02, 50), fontsize=8.5, color=INK2,
            va="center")
for name, vals in beats.items():
    ax.plot(yrs, vals, color=HUES[name], lw=2.4, marker="o", ms=6)
    ax.annotate(f" {name} {vals[-1]:.0f}%", (yrs[-1], vals[-1]),
                color=HUES[name], fontsize=9.5, fontweight="bold",
                va="center")
ax.set_xscale("log")
ax.set_xticks(yrs, [f"{y:.0f}y" for y in yrs])
ax.set_xlim(yrs[0] * 0.85, yrs[-1] * 1.55)
ax.set_ylim(0, 100)
ax.set_xlabel("holding period")
ax.set_ylabel("% of resampled paths that beat SPY")
ax.set_title("Does patience help? — win rate vs holding period",
             loc="left", fontweight="bold")
fig.suptitle(f"{PATHS:,} paths per point, {books.index[0].date()} – "
             f"{books.index[-1].date()} sample, {BOOK_COST_MODEL} costs",
             x=0.01, ha="left", fontsize=10, color=INK2)
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

for name, vals in beats.items():
    d = vals[-1] - vals[0]
    verdict = ("RISES — a persistent edge patience harvests" if d > 2 else
               "FALLS — a persistent deficit time compounds" if d < -2 else
               "FLAT — no edge either way at any horizon")
    print(f"{name}: {vals[0]:.1f}% at 1y -> {vals[-1]:.1f}% at 10y  "
          f"({d:+.1f} pp)  {verdict}")

## 9 · One table, two units — RMX-B and RMX-C as new rows

The requested single view. The `unit` column is not decoration: it is the only
thing keeping this table honest.

* A legacy row is **250 trades at 1% risk each**. Its equity multiple is a
  function of a sizing dial you chose, and its "P(loss)" is a statement about
  the strategy.
* A book row is **252 sessions fully invested**. There is no dial, and its
  "P(loss)" is mostly a statement about the stock market.

The two `final ×` columns are therefore *not* comparable, and a book beating a
sleeve on this table means nothing — turn `RISK_PCT` up and the sleeve rows
move while the book rows do not. What **is** comparable within each block is
the drawdown column, and the only column that speaks to skill for a book is
`P(< SPY)`.

In [ ]:
def _row(name, r, unit, sizing, cost, bench=None):
    return {"unit": unit, "sizing": sizing, "cost bps": cost,
            "n observed": r["n_observed"],
            "final ×  p5": round(r["final_pct"][5], 2),
            "final ×  p50": round(r["final_pct"][50], 2),
            "final ×  p95": round(r["final_pct"][95], 2),
            "maxDD %  p50": round(r["maxdd_pct"][50], 1),
            "maxDD %  p95": round(r["maxdd_pct"][95], 1),
            "P(loss) %": round(r["prob_loss"], 1),
            "P(< SPY) %": bench,
            f"P(DD>{RUIN_DD:.0f}%) %": round(r["prob_ruin"], 1)}

# The cost column has to say what each row actually paid. The legacy rows pay
# one flat number to every trade; a book row pays its own per-name tier spread,
# so its honest figure is the turnover-weighted effective rate, and a benchmark
# that is never traded pays nothing.
def _book_cost(name):
    if BOOK_COST_MODEL == "flat":
        return f"{BOOK_COST_BPS:g}"
    e = meta.get("effective_bps", {}).get(name)
    return f"{e:.2f} eff" if e is not None else "—"

combined = pd.DataFrame(
    [_row(n, mc[n], f"{N_TRADES} trades", f"{RISK_PCT:.1f}%/trade",
          f"{COST_BPS:g}", "—") for n in names]
    + [_row(n, bmc[n], f"{HORIZON} sessions", "fully invested",
            _book_cost(n),
            "—" if np.isnan(bmc[n]["prob_underperform"])
            else round(bmc[n]["prob_underperform"], 1))
       for n in BOOK_SERIES],
    index=names + list(BOOK_SERIES))
combined.index.name = "strategy"
combined

In [ ]:
def _spread(vals, gap):
    """Nudge end-of-line labels apart so they stay readable.

    The four book medians land within 0.01 of each other by construction — they
    are all ~beta-1 equity books — so direct labelling collides unless the text
    is allowed to drift off the point it annotates, with a leader line back.
    """
    order = np.argsort(vals)
    out = np.asarray(vals, dtype=float).copy()
    for i in range(1, len(order)):
        a, b = order[i - 1], order[i]
        if out[b] - out[a] < gap:
            out[b] = out[a] + gap
    return out

fig, (axM, axD) = plt.subplots(1, 2, figsize=(12.5, 5.0))

# left: every median path, both units, on a shared "about a year" x-axis
lbls = names + list(BOOK_SERIES)
curves = {n: fans[n][2] for n in names} | {n: bfans[n][2] for n in BOOK_SERIES}
ends = [curves[n][-1] for n in lbls]
lo_y = min(c.min() for c in curves.values())
hi_y = max(c.max() for c in curves.values())
span = hi_y - lo_y
label_y = _spread(ends, gap=span * 0.062)
for n in names:
    axM.plot(x / N_TRADES, curves[n], color=HUES[n], lw=1.3, alpha=0.7)
for n in BOOK_SERIES:
    axM.plot(bx / HORIZON, curves[n], color=HUES[n], lw=2.4,
             ls="-" if n.startswith("RMX") else (0, (5, 2)))
for lbl, y0, y1 in zip(lbls, ends, label_y):
    bold = lbl.startswith("RMX")
    axM.annotate(lbl, xy=(1.0, y0), xytext=(1.07, y1),
                 color=HUES[lbl], fontsize=9 if bold else 8,
                 fontweight="bold" if bold else "normal", va="center",
                 arrowprops=dict(arrowstyle="-", color=HUES[lbl], lw=0.6,
                                 alpha=0.5, shrinkA=0, shrinkB=2))
axM.axhline(1.0, color=BASELINE, lw=1, ls=(0, (4, 3)))
axM.set_xlim(0, 1.40)
# headroom for the spread labels: matplotlib does not autoscale for annotation
# text, and without this the top three collide with each other and the title
axM.set_ylim(lo_y - 0.03 * span, max(hi_y, label_y.max()) + 0.06 * span)
axM.set_xlabel("fraction of the ~1-year horizon (250 trades / 252 sessions)")
axM.set_ylabel("median equity multiple")
axM.set_title("Median paths — thin = legacy sleeves (sized), thick = books "
              "(fully invested)", loc="left", fontsize=9.5, fontweight="bold",
              pad=10)

# right: the comparable axis. Drawdown is unit-free.
ypos = np.arange(len(lbls))[::-1]
p50 = [mc[n]["maxdd_pct"][50] for n in names] + \
      [bmc[n]["maxdd_pct"][50] for n in BOOK_SERIES]
p95 = [mc[n]["maxdd_pct"][95] for n in names] + \
      [bmc[n]["maxdd_pct"][95] for n in BOOK_SERIES]
for y, lbl, a, b in zip(ypos, lbls, p50, p95):
    axD.plot([a, b], [y, y], color=HUES[lbl], lw=3, alpha=0.45,
             solid_capstyle="round")
    axD.plot([a], [y], "o", color=HUES[lbl], ms=6)
    axD.plot([b], [y], "o", color=HUES[lbl], ms=6, mfc=SURFACE, mew=1.6)
    axD.annotate(f"  {b:.0f}%", (b, y), fontsize=8, color=INK2, va="center")
# the rows above the rule are trade-space, below are session-space. `names` are
# plotted at the HIGH y positions, so the divider sits at len(BOOK_SERIES)-0.5,
# not len(names)-0.5.
split = len(BOOK_SERIES) - 0.5
axD.axhline(split, color=BASELINE, lw=0.8)
axD.annotate("250 trades, sized ↑\n252 sessions, fully invested ↓",
             (5.6, split), fontsize=8, color=MUTED, ha="left", va="center",
             bbox=dict(fc=SURFACE, ec="none", pad=1.5))
axD.set_yticks(ypos, lbls, fontsize=9)
axD.axvline(RUIN_DD, color="#d03b3b", lw=1, ls=(0, (4, 3)))
axD.annotate(f"ruin {RUIN_DD:.0f}%", (RUIN_DD - 0.5, ypos[0] + 0.45),
             fontsize=8.5, color="#d03b3b", ha="right")
axD.set_xlim(5, 55)
axD.set_ylim(-0.8, len(lbls) - 0.2)
axD.set_xlabel("max drawdown % — filled dot p50, hollow p95")
axD.set_title("Drawdown, the one axis both units share", loc="left",
              fontsize=9.5, fontweight="bold", pad=10)
axD.grid(axis="y", visible=False)
fig.suptitle("Legacy sleeves and research-stack books, side by side",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

## Caveats

**Sections 1–6, the legacy sleeves**

- **Bootstrap = i.i.d. resampling.** It scrambles trade order, so regime
  clustering (losing streaks in a bad tape) is *understated*; real drawdowns
  can run deeper than the fans suggest. (Sections 7–9 fix this with blocks —
  it could not be fixed here, because the trade records carry no usable
  session-level ordering across tickers.)
- Trades come from the **vectorized VBT engine** — same known deviations as
  `ablate_vbt.py` (no chandelier trail, ewm-Wilder smoothing, one position
  per ticker). Rankings and shapes are what to trust, not the third decimal.
- Sleeves are simulated **independently and pooled by concatenation**;
  correlation between sleeves trading the same tape is not modeled. A
  portfolio-level sim (`rhengine/portfolio.py`) is the cross-check.
- These are R-multiples, and **R-sums are not returns**. The legacy engine's
  numbers were not re-established after the 2026-07-24 audit; nothing here may
  be quoted as a performance claim.
- The **30 bps is right for these rows.** It is calibrated on the
  `min_dollar_volume = $500k/day` scan universe, which really is small- and
  micro-cap, and ~33 bps really does erase that engine's gross edge.

**Sections 7–9, the research-stack books**

- **One realised sample, re-arranged.** Blocks preserve loss clustering; they
  do not create evidence. 8.5 years cannot make a +1.78%/yr alpha significant
  — that needs ~39.
- The window is **in-sample on purpose** (2010-07 .. 2018-12). Both local
  holdouts are spent (CLAUDE.md rule 4), and re-running these cells on
  2019–2026 would not fix that; it would just spend them again.
- **Cost is per-name, and that was not a detail.** These books are >92%
  mega-cap (median holding ~$19bn), so they pay ~2.8 bps round trip, not 30.
  Charging them the legacy engine's flat 30 overstated cost ~10× and by itself
  flipped RMX-C's measured alpha vs the CW universe from **+0.32% to −0.09%**.
  Set `BOOK_COST_MODEL = "tier-lo"` / `"tier-hi"` and re-run for the bracket;
  the tier table is a published **prior**, not your fills. Research rules §6a,
  and `py scripts/cap_cost_profile.py` for the tier weights.
- `P(loss)` on a book row is a **market** number, not a skill number: RMX-B
  and SPY read within a tenth of a point of each other. The skill column is
  `P(< SPY)` — around 39–44% — which is the documented "expect to trail the
  index roughly 4 years in 10", not a defect.
- Vol-scaling differs between the two books by design: **RMX-B off, RMX-C on**
  (momentum block). That asymmetry is the shipped specification, pinned by
  `tests/test_mc_books.py::test_defaults_match_the_shipped_specification`.
- Every book figure is stamped with `panel_fp`. A number quoted without it is
  not reproducible after the next panel rebuild — see docs/RMX-C.md for the
  one that got away.
- Per the findings spec: whole-history numbers only — promotion decisions
  still require the walk-forward / OOS gates, not this notebook. Neither book
  is `autonomous`; both are `manual` in `strategies.json`.